<a href="https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Santosh-S321/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
os.chdir("/content")
REPO_URL = "https://github.com/Santosh-S321/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
from scipy.stats import spearmanr
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

## 1. Distributions

Checking distributions of key fields before testing anything — traffic-like metrics
are almost always heavy-tailed (a few giant pages, a long tail of tiny ones), which
changes how correlation should be computed below.

In [2]:
for col in ["impressions_90d", "sessions_90d", "word_count", "ctr", "search_volume"]:
    if col in df.columns:
        print(f"{col}: mean={df[col].mean():.1f}, median={df[col].median():.1f}, "
              f"p95={df[col].quantile(0.95):.1f}, max={df[col].max():.1f}")

print("\nHeavy-tail check — mean >> median means a few giants are pulling the average up.")
print("impressions_90d mean/median ratio:", round(df["impressions_90d"].mean() / df["impressions_90d"].median(), 1))

impressions_90d: mean=5200.4, median=731.0, p95=22996.5, max=517715.0
sessions_90d: mean=37.1, median=7.0, p95=166.0, max=4345.0
word_count: mean=3107.8, median=2877.0, p95=6173.0, max=9546.0
ctr: mean=0.5, median=0.1, p95=1.1, max=100.0
search_volume: mean=158.9, median=10.0, p95=390.0, max=74000.0

Heavy-tail check — mean >> median means a few giants are pulling the average up.
impressions_90d mean/median ratio: 7.1


**Note:** All five fields show mean well above median — confirming heavy tails, as
expected for web/traffic metrics. `impressions_90d` is the most extreme: mean 5,200
vs. median 731 (a 7.1x ratio), meaning a small number of giant pages are pulling the
average way up. `search_volume` shows the same pattern (mean 158.9 vs. median 10.0,
a ~16x ratio) — even more skewed. `ctr` is also notable: median 0.1 but max 100.0,
consistent with the flyrank-data warning that some rate columns can exceed normal
bounds. Given this, using Spearman (rank-based) correlation and log1p-transformed
Pearson for Test 1 below, not raw Pearson — raw correlation on these heavy-tailed
values would be dominated by the handful of giant pages.

## 2. Signal test #1 / #2 / #3 (verdict each)

**Test 1 — "High search volume means more traffic."** Claim: search_volume predicts
impressions_90d. Test: Spearman correlation (rank-based, handles heavy tails) plus a
log1p Pearson check.

**Test 2 — "Longer content ranks better."** Claim: higher word_count → better
avg_position. Test: grouped median avg_position by word_count tier, with n shown.

**Test 3 — "Higher CPC keywords perform better organically."** Claim: higher cpc →
more impressions. Test: grouped median impressions_90d by cpc tier, with n shown.

In [3]:
# Test 1 — search volume vs impressions (fixed: drop NaNs first)
valid = df[["search_volume", "impressions_90d"]].dropna()
spearman_corr, _ = spearmanr(valid["search_volume"], valid["impressions_90d"])
pearson_log_corr = np.corrcoef(np.log1p(valid["search_volume"]), np.log1p(valid["impressions_90d"]))[0, 1]
print(f"Test 1 — Spearman: {spearman_corr:.3f} | log1p Pearson: {pearson_log_corr:.3f} | n={len(valid)}")

# Test 2 — word count vs position
df["word_count_tier"] = pd.qcut(df["word_count"], q=5, duplicates="drop")
t2 = df.groupby("word_count_tier", observed=True)["avg_position"].agg(["median", "count"])
print("\nTest 2 — median avg_position by word_count tier:")
print(t2)

# Test 3 — CPC vs impressions
if "cpc" in df.columns:
    df["cpc_tier"] = pd.qcut(df["cpc"], q=5, duplicates="drop")
    t3 = df.groupby("cpc_tier", observed=True)["impressions_90d"].agg(["median", "count"])
    print("\nTest 3 — median impressions_90d by CPC tier:")
    print(t3)
else:
    print("\nTest 3 — 'cpc' column not found; check column name before finalizing.")

Test 1 — Spearman: -0.029 | log1p Pearson: -0.026 | n=27532

Test 2 — median avg_position by word_count tier:
                  median  count
word_count_tier                
(7.999, 1725.0]      8.1   4462
(1725.0, 2724.0]     9.5   4461
(2724.0, 3060.0]     9.1   4465
(3060.0, 3973.0]    10.9   4453
(3973.0, 9546.0]    16.6   4460

Test 3 — median impressions_90d by CPC tier:
                median  count
cpc_tier                     
(-0.001, 0.11]   988.0  22077
(0.11, 100.36]   704.0   5455


**Test 1 verdict: FALSE.** Spearman -0.029, log1p Pearson -0.026, n=27,532 (2,468 rows
dropped for missing search_volume). Essentially no relationship — replicates
Notebook 1's Discovery A (0.001) almost exactly, and matches the paper's own Myth 1
finding directionally (search volume behaves as a competition signal, not a traffic
forecast).

**Test 2 verdict: OPPOSITE.** Median avg_position worsens monotonically as word count
rises: 8.1 → 9.5 → 9.1 → 10.9 → 16.6 across five tiers, all n≈4,460+. Longer content
ranks worse, not better, in this dataset. Note: this measures position, not impressions
or query-count — the paper's own Myth 3 (CONFIRMED) used different outcome variables,
so these findings aren't necessarily contradictory.

**Test 3 verdict: OPPOSITE.** Low-CPC pages outperform (988 median impressions,
n=22,077) vs. high-CPC pages (704, n=5,455). Matches the paper's Myth 8 finding
(OPPOSITE) exactly. Note: qcut collapsed to 2 tiers instead of 5 due to many pages
sharing CPC=0 — a data-shape note, not a bug.

## 3. The flag-linked test

**Claim (behind FlyRank's `is_quick_win`-style logic):** higher-impression pages are
more likely to sit in "striking distance" (position 11-20) — the highest-ROI zone,
since a small improvement there yields fast wins.

**Test:** grouped share of pages in striking distance, by impressions tier, with n shown.

In [4]:
df["impressions_tier"] = pd.qcut(df["impressions_90d"], q=5, duplicates="drop")
df["in_striking_distance"] = df["position_tier"].eq("striking").astype(int)
t4 = df.groupby("impressions_tier", observed=True)["in_striking_distance"].agg(["mean", "count"])
print("Share of pages in striking distance, by impressions tier:")
print(t4)

Share of pages in striking distance, by impressions tier:
                        mean  count
impressions_tier                   
(0.999, 39.0]       0.149810   6041
(39.0, 364.0]       0.256875   5964
(364.0, 1375.0]     0.316658   5997
(1375.0, 5167.6]    0.311604   5998
(5167.6, 517715.0]  0.183167   6000


**Verdict: MIXED.** Striking-distance share rises from 0.150 → 0.257 → 0.317 → 0.312
across the first four impressions tiers, then drops to 0.183 at the highest tier — a
genuine reversal, not a sample-size artifact (all buckets n≈6,000). Very high-traffic
pages are actually less likely to sit in striking distance, plausibly because they've
already broken through to page 1.

## 4. What this means in practice

Three of four beliefs tested came back FALSE or OPPOSITE of the common assumption:
search volume doesn't predict traffic, longer content doesn't rank better, and
cheaper keywords outperform expensive ones — all consistent with the paper's own myth
tests. The flag-linked striking-distance test was MIXED: traffic helps a page reach
striking distance up to a point, but the very highest-traffic pages are less likely to
be stuck there, since they've typically already broken through. Practical takeaway for
a content team: don't chase word count or search-volume as prioritization signals —
they're not correlated with what actually matters here. Impressions tier is a useful
but non-linear signal for identifying striking-distance opportunities.

In [5]:
print("Summary written above — no additional computation needed.")

Summary written above — no additional computation needed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.